# **ANÁLISIS DE POSICIONAMIENTO EN YOUTUBE**


## 1. Contexto

El posicionamiento en plataformas como YouTube representa una oportunidad para entender las preferencias de la audiencia y encontrar patrones de consumo. El objetivo es analizar los videos que fueron tendencia en Estados Unidos, con el fin de identificar patrones asociados con su popularidad, la interacción de los usuarios, el comportamiento, las diferencias entre categorías y las visualizaciones.

A partir de un análisis exploratorio y mediante la aplicación de la ciencia de datos, el propósito es encontrar información que permita entender mejor el comportamiento de los videos en tendencia y, con ello, tomar decisiones.

### Objetivos específicos

* Identificar las categorías y los canales con mayor presencia en Trending.
* Analizar el comportamiento de las visualizaciones y las interacciones de los usuarios.
* Estudiar cuánto tiempo transcurre entre la publicación de un video y su primera aparición en Trending.
* Identificar patrones de comportamiento entre diferentes tipos de contenido.
* Evaluar la posibilidad de desarrollar modelos que permitan caracterizar o estimar el comportamiento de los videos.

## 2. Importación de datos y análisis inicial

In [ ]:
from pathlib import Path
import sys
from utils.kaggle_utils import import_data

ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR))

us_videos = import_data('USvideos.csv')
us_category_id = import_data('US_category_id.json')


/home/json/Documents/pragma_test


/home/json/.local/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from IPython.display import display
import warnings
warnings.filterwarnings("ignore")

print("USvideos size:", us_videos.shape)
print("USvideos size unique rows:", us_videos.drop_duplicates(inplace=False).shape)
print("US_category_id size:", us_category_id.shape)
print("_____________________________________________________________")
print("USvideos columns:", us_videos.columns.tolist())
print("US_category_id columns:", us_category_id.columns.tolist())
print("_____________________________________________________________")
display(us_videos.head())
display(us_category_id.head())
print("_____________________________________________________________")
us_videos.info()
us_category_id.info()
print("_____________________________________________________________")
us_videos.drop_duplicates(inplace=False).isnull().sum()



In [ ]:
us_category_id.isnull().sum()

En el análisis inicial se revisó la información de cada tabla:

**USvideos.csv:** Contiene 16 columnas y 40.949 registros, aunque hay 48 duplicados en todas sus columnas, por lo que la cantidad final de filas es 40.901. Esta tabla contiene datos como `video_id`, `trending_date`, `title`, `channel_title`, `category_id`, `publish_time`, `tags`, `views`, `likes`, `dislikes`, `comment_count`, `thumbnail_link`, `comments_disabled`, `ratings_disabled`, `video_error_or_removed` y `description`, cuyos nombres son bastante dicientes sobre los datos que los componen. Los tipos de datos deben ser casteados para su correcto uso y solo se encontraron 570 valores nulos en la columna `description`.

**US_category_id.json:** Contiene el catálogo de categorías de YouTube en formato JSON. Incluye información como el identificador de cada categoría (`id`), su nombre (`title`) y un atributo que indica si la categoría puede asignarse (`assignable`). Esta información permite interpretar los valores de `category_id` presentes en los registros de videos y facilitar su clasificación y análisis por categoría.

Para el análisis de los datos se utilizará PostgreSQL, ya que facilita las consultas y la transformación de datos, particularmente en las funciones de agregación, operaciones con fechas y análisis temporal. Además se parseara el json de la tabla US_category_id antes de cargarla en la base de datos.

In [ ]:
import pandas as pd
from postgres_utils import load_dataframe_to_postgres

us_category_id_parsed = pd.json_normalize(
    us_category_id["items"].tolist(),
    sep="_",
).rename(columns={
    "snippet_title": "title",
    "snippet_assignable": "assignable",
})[["id", "title", "assignable"]]
us_category_id_parsed["id"] = us_category_id_parsed["id"].astype("int64")

us_videos_to_load = us_videos.drop_duplicates().copy()
us_videos_to_load["trending_date"] = pd.to_datetime(
    us_videos_to_load["trending_date"], format="%y.%d.%m"
).dt.date
us_videos_to_load["publish_time"] = pd.to_datetime(
    us_videos_to_load["publish_time"], utc=True
).dt.tz_localize(None)


#load_dataframe_to_postgres("us_videos", us_videos_to_load)
#load_dataframe_to_postgres("us_category_id", us_category_id_parsed)